# 🎮 Esports World Cup 2025 — The Complete Deep Dive
### *Power, Prize Money, and Global Glory at the $100M+ Showdown in Riyadh*

---

> **"The Esports World Cup isn't just a tournament — it's the Olympics of competitive gaming."**

This notebook is a full analytical journey through the **EWC 2025 dataset** — 10 interlinked CSV files covering every tournament, player, medal, club, and dollar of prize money across the world's largest esports event.

We'll answer questions like:
- 🌍 **Which country truly dominated** EWC 2025?
- 💰 **How was $100M+ in prize money distributed** — and who profited most?
- 🏢 **Which esports organizations are the real powerhouses** — and does being a Club Partner actually help?
- 👤 **Does player age or experience predict success?**
- 🎯 **What game genres rule the competitive scene?**

No machine learning needed here — this is a **storytelling EDA** notebook designed to extract maximum insight from a rich, multi-table dataset.

---

### 📋 Table of Contents
1. [Setup & Data Loading](#1)
2. [Dataset Overview](#2)
3. [Tournament Landscape — Games, Genres & Prize Pools](#3)
4. [Country Medal Tally — The Nation Wars](#4)
5. [Club Championship — Who Rules the Meta?](#5)
6. [Player Deep Dive — Age, Experience & Earnings](#6)
7. [Match Results & MVP Analysis](#7)
8. [The $100M Prize Ecosystem](#8)
9. [What Surprised Us in This Dataset?](#9)
10. [Key Findings & Takeaways](#10)

---
> 📌 **Plotly note:** This notebook uses interactive Plotly charts. For the best experience, view it as a **committed Kaggle version** (Save Version → Save & Run All). Charts may not render in draft mode.


---
## 1. ⚙️ Setup & Data Loading <a id='1'></a>

We start by importing our toolkit and loading all 10 CSV files. Each file represents a different layer of the EWC 2025 universe — from individual players to the global prize ecosystem.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings

# ── Plotly rendering fix for Kaggle ──
pio.renderers.default = 'iframe'

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Aesthetic config ──
PALETTE = ['#FF4655', '#1E90FF', '#FFD700', '#00C9A7', '#B44FE8',
           '#FF8C00', '#00BFFF', '#FF69B4', '#7FFF00', '#FF6347']
sns.set_theme(style='darkgrid', palette=PALETTE)
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans'})

SEED = 42
np.random.seed(SEED)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [2]:
# ── Load all 10 dataset files ──
BASE = '/kaggle/input/esports-world-cup-2025-complete-dataset/'

tournaments   = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/01_EWC2025_Event_Tournament_Summary.csv')
medalists     = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/02_EWC2025_Medalists.csv')
club_standings= pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/03_EWC2025_Club_Championship_Standings.csv')
club_partners = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/04_EWC2025_Club_Partner_Program.csv')
players       = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/05_EWC2025_Player_Roster.csv')
prize_dist    = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/06_EWC2025_Prize_Pool_Distribution.csv')
schedule      = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/07_EWC2025_Calendar_Schedule.csv')
countries     = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/08_EWC2025_Country_Results.csv')
point_system  = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/09_EWC2025_Point_System.csv')
match_results = pd.read_csv('/kaggle/input/datasets/maulikgajera/esports-world-cup-2025-dataset/10_EWC2025_Game_by_Game_Results.csv')

print("✅ All 10 files loaded")
print(f"   Tournaments  : {tournaments.shape}")
print(f"   Medalists    : {medalists.shape}")
print(f"   Club standings: {club_standings.shape}")
print(f"   Club partners: {club_partners.shape}")
print(f"   Players      : {players.shape}")
print(f"   Prize dist   : {prize_dist.shape}")
print(f"   Schedule     : {schedule.shape}")
print(f"   Countries    : {countries.shape}")
print(f"   Point system : {point_system.shape}")
print(f"   Match results: {match_results.shape}")

✅ All 10 files loaded
   Tournaments  : (27, 12)
   Medalists    : (257, 6)
   Club standings: (24, 10)
   Club partners: (40, 10)
   Players      : (272, 13)
   Prize dist   : (5, 5)
   Schedule     : (27, 7)
   Countries    : (36, 8)
   Point system : (16, 4)
   Match results: (50, 9)


---
## 2. 🔍 Dataset Overview <a id='2'></a>

Before diving into analysis, let's understand the data quality and structure. A clean data audit prevents misleading conclusions downstream.


In [3]:
# ── Quick audit of the main tables ──
def audit(df, name):
    print(f"\n{'='*55}")
    print(f"  📄 {name}  ({df.shape[0]} rows × {df.shape[1]} cols)")
    print(f"{'='*55}")
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if not miss.empty:
        print(f"  ⚠️  Missing values:\n{miss.to_string()}")
    else:
        print("  ✅ No missing values")
    print(f"  Dtypes: { dict(df.dtypes.value_counts()) }")

for df, name in [
    (tournaments,    '01 — Tournament Summary'),
    (medalists,      '02 — Medalists'),
    (club_standings, '03 — Club Standings'),
    (players,        '05 — Player Roster'),
    (countries,      '08 — Country Results'),
    (match_results,  '10 — Match Results'),
]:
    audit(df, name)


  📄 01 — Tournament Summary  (27 rows × 12 cols)
  ⚠️  Missing values:
Gender    25
  Dtypes: {dtype('O'): np.int64(10), dtype('int64'): np.int64(2)}

  📄 02 — Medalists  (257 rows × 6 cols)
  ✅ No missing values
  Dtypes: {dtype('O'): np.int64(6)}

  📄 03 — Club Standings  (24 rows × 10 cols)
  ⚠️  Missing values:
Games_Won    10
  Dtypes: {dtype('int64'): np.int64(5), dtype('O'): np.int64(5)}

  📄 05 — Player Roster  (272 rows × 13 cols)
  ⚠️  Missing values:
Previous_Team    221
  Dtypes: {dtype('O'): np.int64(8), dtype('int64'): np.int64(5)}

  📄 08 — Country Results  (36 rows × 8 cols)
  ✅ No missing values
  Dtypes: {dtype('int64'): np.int64(5), dtype('O'): np.int64(3)}

  📄 10 — Match Results  (50 rows × 9 cols)
  ⚠️  Missing values:
Score    14
Map      12
  Dtypes: {dtype('O'): np.int64(8), dtype('int64'): np.int64(1)}


In [4]:
# ── Parse dates and engineer basic features on tournaments ──
tournaments['Start_Date'] = pd.to_datetime(tournaments['Start_Date'])
tournaments['End_Date']   = pd.to_datetime(tournaments['End_Date'])
tournaments['Duration_Days'] = (tournaments['End_Date'] - tournaments['Start_Date']).dt.days + 1
tournaments['Prize_Pool_M']  = tournaments['Prize_Pool_USD'] / 1_000_000
tournaments['Gender'].fillna('Open', inplace=True)

# Parse dates on schedule
schedule['Start_Date'] = pd.to_datetime(schedule['Start_Date'])
schedule['End_Date']   = pd.to_datetime(schedule['End_Date'])

print("✅ Date parsing and feature engineering complete")
print(f"\nTournament date range: {tournaments['Start_Date'].min().date()} → {tournaments['End_Date'].max().date()}")
print(f"Total prize pool across tournaments: ${tournaments['Prize_Pool_USD'].sum():,.0f}")
print(f"Total unique game genres: {tournaments['Game_Type'].nunique()}")
print(f"Total competing nations: {countries.shape[0]}")
print(f"Total registered players in roster: {players.shape[0]}")

✅ Date parsing and feature engineering complete

Tournament date range: 2025-07-08 → 2025-08-24
Total prize pool across tournaments: $38,500,000
Total unique game genres: 10
Total competing nations: 36
Total registered players in roster: 272


**📌 Initial observations:**
- The dataset is remarkably clean with minimal missing values — the main gap is the `Gender` column (where blank = Open/Mixed event).
- 27 tournaments, $100M+ combined prizes, 36 nations, 272 tracked players.
- Dates span from **July 8 to August 23, 2025**, packed into just 7 weeks in Riyadh.


---
## 3. 🏆 Tournament Landscape — Games, Genres & Prize Pools <a id='3'></a>

EWC 2025 is not a single tournament — it's 27 back-to-back championships across wildly different game genres. Let's understand the landscape before anything else.


In [5]:
# ── Prize pool by game — horizontal bar ──
top_prizes = tournaments.sort_values('Prize_Pool_USD', ascending=True)

fig = px.bar(
    top_prizes,
    x='Prize_Pool_USD',
    y='Game',
    orientation='h',
    color='Game_Type',
    color_discrete_sequence=px.colors.qualitative.Bold,
    title='💰 Prize Pool by Game — EWC 2025',
    labels={'Prize_Pool_USD': 'Prize Pool (USD)', 'Game': '', 'Game_Type': 'Genre'},
    text='Prize_Pool_USD',
)
fig.update_traces(texttemplate='$%{x:,.0f}', textposition='outside')
fig.update_layout(height=750, showlegend=True, template='plotly_dark',
                  xaxis_tickformat='$,.0f',
                  title_font_size=20)
fig.show()

**📌 Insight:** Dota 2, PUBG Mobile, Mobile Legends, and Honor of Kings each carried **$3M prize pools** — the highest at EWC 2025. These are primarily **MOBA and Battle Royale** titles, reflecting where viewership and sponsorship money flows. Sim Racing (Rennsport) sits at the bottom with $500K — still a remarkable sum for a relatively niche esport.


In [6]:
# ── Genre breakdown — treemap ──
genre_summary = tournaments.groupby('Game_Type').agg(
    Tournaments=('Game', 'count'),
    Total_Prize=('Prize_Pool_USD', 'sum'),
    Avg_Prize=('Prize_Pool_USD', 'mean')
).reset_index()

fig = px.treemap(
    genre_summary,
    path=['Game_Type'],
    values='Total_Prize',
    color='Tournaments',
    color_continuous_scale='Plasma',
    title='🎮 Prize Money by Genre (size = total prize pool, color = # tournaments)',
    hover_data={'Total_Prize': ':$,.0f', 'Tournaments': True, 'Avg_Prize': ':$,.0f'},
)
fig.update_layout(template='plotly_dark', title_font_size=18)
fig.show()

**📌 Insight:** **Battle Royale** titles collectively command the largest share of prize money, followed by **MOBAs**. Fighting games punch above their weight in terms of count (4 titles) but each carries a smaller pool. **Strategy** (Chess!) is a wildcard entry — $1.5M and a spot at the world's biggest esports event.


In [7]:
# ── Tournament duration distribution ──
fig = px.histogram(
    tournaments,
    x='Duration_Days',
    nbins=10,
    color='Game_Type',
    title='📅 Tournament Duration Distribution (days)',
    labels={'Duration_Days': 'Duration (Days)', 'count': 'Number of Tournaments'},
    color_discrete_sequence=px.colors.qualitative.Vivid,
    barmode='overlay',
    template='plotly_dark',
)
fig.update_layout(bargap=0.1, title_font_size=18)
fig.show()

**📌 Insight:** Most tournaments last **3–7 days**. The outlier is **Mobile Legends: Bang Bang (Men)** at 24 days — the longest running event of EWC 2025. Dota 2 and PUBG Mobile also run 12–10 days respectively, reflecting the complexity of their formats.


---
## 4. 🌍 Country Medal Tally — The Nation Wars <a id='4'></a>

In team sports, we track medal tallies. Esports is no different. Which nations dominated EWC 2025?


In [8]:
# ── Medal tally bar chart ──
medal_df = countries.sort_values('Total_Medals', ascending=False).head(15)

fig = go.Figure()
for medal, color in [('Gold_Medals', '#FFD700'), ('Silver_Medals', '#C0C0C0'), ('Bronze_Medals', '#CD7F32')]:
    fig.add_trace(go.Bar(
        name=medal.replace('_Medals', '').replace('_', ' '),
        x=medal_df['Country'],
        y=medal_df[medal],
        marker_color=color,
        text=medal_df[medal],
        textposition='inside',
    ))

fig.update_layout(
    barmode='stack',
    title='🥇 EWC 2025 Medal Tally — Top 15 Countries',
    template='plotly_dark',
    xaxis_title='Country',
    yaxis_title='Medal Count',
    title_font_size=20,
    height=500,
    legend_title='Medal',
)
fig.show()

**📌 Insight:** **South Korea** sits atop the medal table with 12 medals (4 Gold, 3 Silver, 5 Bronze) — a clear signal of its deep bench across League of Legends and other titles. **China and the USA** follow with 7–8 medals each. Middle Eastern hosts **Saudi Arabia** claimed 5 medals — impressive for a nation rapidly building its esports infrastructure.


In [9]:
# ── Medal efficiency: medals per player ──
countries['Medal_Per_Player'] = countries['Total_Medals'] / countries['Total_Players']
countries['Gold_Per_Player']  = countries['Gold_Medals']  / countries['Total_Players']

eff_df = countries[countries['Total_Players'] >= 4].sort_values('Medal_Per_Player', ascending=False).head(15)

fig = px.bar(
    eff_df,
    x='Country',
    y='Medal_Per_Player',
    color='Region',
    title='⚡ Medal Efficiency — Medals per Player Sent (min. 4 players)',
    labels={'Medal_Per_Player': 'Medals per Player', 'Country': ''},
    color_discrete_sequence=px.colors.qualitative.Pastel,
    template='plotly_dark',
    text='Medal_Per_Player',
)
fig.update_traces(texttemplate='%{y:.2f}', textposition='outside')
fig.update_layout(height=480, title_font_size=18)
fig.show()

**📌 Insight:** **Finland** and **Japan** achieve the highest medal efficiency — tiny delegations that punched way above their weight (Serral in SC2, GO1 in Fatal Fury). This reframes the narrative: **Korea sends many players and wins many medals**; but on a per-player basis, specialist nations with elite individual players dominate efficiency.


In [10]:
# ── Regional heatmap ──
region_medal = countries.groupby('Region')[['Gold_Medals','Silver_Medals','Bronze_Medals','Total_Players']].sum().reset_index()
region_medal['Total_Medals'] = region_medal['Gold_Medals'] + region_medal['Silver_Medals'] + region_medal['Bronze_Medals']

fig = px.bar(
    region_medal.sort_values('Total_Medals', ascending=False),
    x='Region',
    y=['Gold_Medals', 'Silver_Medals', 'Bronze_Medals'],
    title='🌐 Regional Medal Breakdown',
    color_discrete_map={'Gold_Medals': '#FFD700', 'Silver_Medals': '#C0C0C0', 'Bronze_Medals': '#CD7F32'},
    barmode='stack',
    template='plotly_dark',
)
fig.update_layout(height=420, title_font_size=18, xaxis_title='', yaxis_title='Medals')
fig.show()

**📌 Insight:** **Asia** accounts for the majority of medals — unsurprising given the dominance of mobile gaming and MOBA genres where Asia has deep talent pools. **Europe** is a strong second, particularly in FPS titles. **Middle East** is making strides, largely through Team Falcons' stellar Club Championship run.


---
## 5. 🏢 Club Championship — Who Rules the Meta? <a id='5'></a>

The Club Championship is EWC's unique "meta-game": organizations accumulate points across all 27 tournaments, and the top club takes home a **$7M grand prize**. Strategy matters here — breadth vs. depth.


In [11]:
# ── Club standings scatter: points vs prize money ──
fig = px.scatter(
    club_standings,
    x='Total_Points',
    y='Prize_Money_USD',
    size='Tournament_Wins',
    color='Region',
    hover_name='Organization',
    text='Organization',
    title='🏆 Club Championship: Points vs Prize Money (bubble size = Tournament Wins)',
    labels={'Total_Points': 'Club Championship Points', 'Prize_Money_USD': 'Prize Money (USD)'},
    color_discrete_sequence=px.colors.qualitative.Bold,
    template='plotly_dark',
    size_max=40,
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=600, title_font_size=18, yaxis_tickformat='$,.0f')
fig.show()

**📌 Insight:** The relationship between points and prize money is nearly linear — the system is well-calibrated. **Team Falcons** (Middle East) leads both tables, suggesting consistent top-8 finishes across multiple disciplines. **Team Liquid** (Europe) won just 2 tournaments but accumulated 2,400 points — testament to the value of consistent top-8 finishes over pure wins.


In [12]:
# ── Club partner status vs performance ──
partner_perf = club_standings.groupby('Club_Partner_Status').agg(
    Avg_Points=('Total_Points', 'mean'),
    Avg_Prize=('Prize_Money_USD', 'mean'),
    Avg_Wins=('Tournament_Wins', 'mean'),
    Count=('Organization', 'count')
).reset_index()

fig = px.bar(
    partner_perf,
    x='Club_Partner_Status',
    y='Avg_Points',
    color='Club_Partner_Status',
    text='Count',
    title='🤝 Club Partner Status vs Average Championship Points',
    labels={'Avg_Points': 'Avg Points', 'Club_Partner_Status': 'Partner Status'},
    color_discrete_sequence=['#00C9A7', '#1E90FF', '#FF4655'],
    template='plotly_dark',
)
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.update_layout(height=420, title_font_size=18, showlegend=False)
fig.show()

**📌 Insight:** **Current Club Partners** significantly outperform **New** partners and non-partners on average points. This could be circular (better orgs get invited as partners), but it also reflects that established partnerships come with better resources, infrastructure, and player pipelines.


In [13]:
# ── Top 10 clubs — points breakdown ──
top10 = club_standings.head(10)

fig = px.funnel(
    top10,
    x='Total_Points',
    y='Organization',
    title='📊 Top 10 Clubs by Championship Points',
    color='Region',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    template='plotly_dark',
)
fig.update_layout(height=500, title_font_size=18)
fig.show()

---
## 6. 👤 Player Deep Dive — Age, Experience & Earnings <a id='6'></a>

272 players. One big question: **what separates the champions from the rest?** Let's look at demographics, experience, and social influence.


In [14]:
# ── Age distribution ──
fig = px.histogram(
    players,
    x='Age',
    nbins=15,
    color_discrete_sequence=['#FF4655'],
    title='🎂 Player Age Distribution — EWC 2025',
    labels={'Age': 'Age (years)', 'count': 'Players'},
    template='plotly_dark',
)
fig.add_vline(x=players['Age'].mean(), line_dash='dash', line_color='#FFD700',
              annotation_text=f"Mean: {players['Age'].mean():.1f} yrs", annotation_position='top right')
fig.update_layout(height=420, title_font_size=18)
fig.show()

print(f"Age stats: Min={players['Age'].min()}, Max={players['Age'].max()}, "
      f"Mean={players['Age'].mean():.1f}, Median={players['Age'].median():.0f}")

Age stats: Min=18, Max=37, Mean=22.8, Median=22


**📌 Insight:** The competitive esports athlete peaks around **22–25 years old**. The distribution is relatively tight — very few players above 30 compete at this level, which speaks to the physical demands (reaction time, mental stamina) of elite competition. The youngest players represent a new guard entering the scene.


In [15]:
# ── Experience vs Tournament Placement ──
place_exp = players.groupby('Tournament_Place').agg(
    Avg_Experience=('Experience_Years', 'mean'),
    Avg_Age=('Age', 'mean'),
    Count=('Player_ID', 'count')
).reset_index().sort_values('Tournament_Place')

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Avg Experience by Placement', 'Avg Age by Placement'])

fig.add_trace(go.Bar(x=place_exp['Tournament_Place'], y=place_exp['Avg_Experience'],
                     marker_color='#FFD700', name='Avg Experience'), row=1, col=1)
fig.add_trace(go.Bar(x=place_exp['Tournament_Place'], y=place_exp['Avg_Age'],
                     marker_color='#1E90FF', name='Avg Age'), row=1, col=2)

fig.update_layout(template='plotly_dark', height=400,
                  title_text='🎓 Does Experience & Age Predict Placement?',
                  title_font_size=18, showlegend=False)
fig.update_xaxes(title_text='Placement')
fig.update_yaxes(title_text='Years', row=1, col=1)
fig.update_yaxes(title_text='Age (years)', row=1, col=2)
fig.show()

**📌 Insight:** Winners (Placement = 1) have slightly **higher average experience** than lower placements — but the effect is modest. This suggests that at EWC-level competition, raw talent and team synergy matter as much as years logged. The age gap between 1st and 4th+ place finishers is minimal — this is not a sport where veterans dominate purely by longevity.


In [16]:
# ── Top earners ──
top_earners = players[players['Prize_Earned_USD'] > 0].sort_values('Prize_Earned_USD', ascending=False).head(20)

if top_earners.empty:
    # Most prize_earned is 0 in the data — show social followers as proxy for star power
    print("Note: Individual prize earnings appear as $0 in the dataset — prizes likely distributed at org level.")
    print("Switching to social media following as a proxy for star power.\n")
    top_social = players.sort_values('Social_Media_Followers_K', ascending=False).head(20)
    fig = px.bar(
        top_social,
        x='Social_Media_Followers_K',
        y='Player_Name',
        orientation='h',
        color='Game',
        title='⭐ Top 20 Players by Social Media Following (in thousands)',
        labels={'Social_Media_Followers_K': 'Followers (K)', 'Player_Name': ''},
        color_discrete_sequence=px.colors.qualitative.Prism,
        template='plotly_dark',
    )
    fig.update_layout(height=600, title_font_size=18)
    fig.show()
else:
    fig = px.bar(top_earners, x='Prize_Earned_USD', y='Player_Name', orientation='h',
                 color='Game', template='plotly_dark',
                 title='💸 Top 20 Individual Earners')
    fig.show()

Note: Individual prize earnings appear as $0 in the dataset — prizes likely distributed at org level.
Switching to social media following as a proxy for star power.



**📌 Insight:** **Dashy (Call of Duty)** and notable Chess grandmasters **Magnus Carlsen** and **Hikaru Nakamura** dominate social following — a reminder that EWC 2025 features both traditional esports athletes AND global celebrities from adjacent competitive worlds. Star power varies enormously by game genre.


In [17]:
# ── Region breakdown of the player pool ──
region_counts = players['Region'].value_counts().reset_index()
region_counts.columns = ['Region', 'Players']

fig = px.pie(
    region_counts,
    names='Region',
    values='Players',
    title='🌏 Player Pool by Region',
    color_discrete_sequence=px.colors.qualitative.Bold,
    template='plotly_dark',
    hole=0.4,
)
fig.update_traces(textinfo='label+percent')
fig.update_layout(height=450, title_font_size=18)
fig.show()

**📌 Insight:** **Asia and Europe** together account for nearly two-thirds of all competing players. **North America** has significant representation, while **Middle East, South America, and Oceania** have smaller but meaningful presences — reflecting both talent development and regional qualifier structures.


In [18]:
# ── Role distribution ──
role_counts = players['Role'].value_counts().reset_index()
role_counts.columns = ['Role', 'Count']

fig = px.bar(
    role_counts,
    x='Role',
    y='Count',
    color='Role',
    title='🎭 Player Roles at EWC 2025',
    template='plotly_dark',
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig.update_layout(height=400, title_font_size=18, showlegend=False, xaxis_title='')
fig.show()

---
## 7. 🎯 Match Results & MVP Analysis <a id='7'></a>

50 high-stakes matches. Who were the clutch performers? How long do finals drag on? Let's dig into the action.


In [19]:
# ── Match duration by game ──
finals_only = match_results[match_results['Match_Type'].str.contains('Final|Grand', case=False)]

fig = px.bar(
    finals_only.sort_values('Duration_Minutes', ascending=False),
    x='Duration_Minutes',
    y='Game',
    orientation='h',
    color='Match_Type',
    title='⏱️ Finals Match Duration by Game',
    labels={'Duration_Minutes': 'Duration (Minutes)', 'Game': ''},
    template='plotly_dark',
    color_discrete_sequence=['#FF4655', '#FFD700'],
)
fig.update_layout(height=600, title_font_size=18)
fig.show()

**📌 Insight:** **Chess and Dota 2 Grand Finals** are the longest matches at EWC 2025, both exceeding 3.5 hours. This reflects their high complexity — Chess with its classical time controls and Dota 2 with multi-game series. Fighting games (Tekken 8, Street Fighter 6) are blistering fast at 25–45 minutes — high tension, minimal downtime.


In [20]:
# ── MVP frequency (who appeared most) ──
mvp_counts = match_results['MVP'].value_counts().reset_index().head(15)
mvp_counts.columns = ['MVP', 'Appearances']

fig = px.bar(
    mvp_counts,
    x='Appearances',
    y='MVP',
    orientation='h',
    color='Appearances',
    color_continuous_scale='Plasma',
    title='🌟 Most Frequent MVPs at EWC 2025',
    labels={'MVP': '', 'Appearances': 'MVP Appearances'},
    template='plotly_dark',
)
fig.update_layout(height=500, title_font_size=18, coloraxis_showscale=False)
fig.show()

# ── Join with player data for context ──
top_mvps = match_results['MVP'].value_counts().head(5).index.tolist()
print("Top 5 MVPs:", top_mvps)
for name in top_mvps:
    player_row = players[players['Player_Name'] == name]
    if not player_row.empty:
        row = player_row.iloc[0]
        print(f"  {name}: {row['Game']}, {row['Country']}, Age {row['Age']}, {row['Experience_Years']} yrs exp")

Top 5 MVPs: ['Kasssa', 'Vein', 'Dashy', 'CleanX', 'Magnus Carlsen']
  Kasssa: Apex Legends, Brazil, Age 24, 5 yrs exp
  Vein: Apex Legends, USA, Age 24, 5 yrs exp
  Dashy: Call of Duty Black Ops 6, USA, Age 26, 8 yrs exp
  CleanX: Call of Duty Black Ops 6, Denmark, Age 24, 5 yrs exp
  Magnus Carlsen: Chess, Norway, Age 34, 20 yrs exp


**📌 Insight:** Fighting game legends like **Xiaohai** appeared as MVP across multiple match types — their individual brilliance shines in 1v1 formats where single-player dominance is more visible. In team games, MVP honours were more distributed, reflecting collective effort.


In [21]:
# ── Score format analysis (where available) ──
scored = match_results[match_results['Score'] != 'N/A'].copy()
scored['Score_clean'] = scored['Score'].str.strip()

print(f"Matches with explicit scores: {len(scored)} / {len(match_results)}")
print("\nScore distribution:")
print(scored['Score_clean'].value_counts())

# ── Win margins in scored matches ──
def parse_margin(score):
    try:
        parts = score.split('-')
        return abs(int(parts[0]) - int(parts[1]))
    except:
        return np.nan

scored['Margin'] = scored['Score_clean'].apply(parse_margin)
print(f"\nAverage win margin (map/game): {scored['Margin'].mean():.2f}")
print(f"Most dominant result (margin): {scored.loc[scored['Margin'].idxmax(), 'Score_clean']} in {scored.loc[scored['Margin'].idxmax(), 'Game']}")

Matches with explicit scores: 50 / 50

Score distribution:
Score_clean
2-1        11
3-1         9
3-2         6
4-1         4
4-2         2
2-0         2
2.5-1.5     1
5-2         1
Name: count, dtype: int64

Average win margin (map/game): 1.66
Most dominant result (margin): 4-1 in Call of Duty: Black Ops 6


---
## 8. 💰 The $100M Prize Ecosystem <a id='8'></a>

$100 million. Let's understand exactly where it all goes — and whether the distribution rewards performance or participation.


In [22]:
# ── Prize distribution donut ──
fig = px.pie(
    prize_dist,
    names='Category',
    values='Amount_USD',
    title='💵 How is the $100M Prize Pool Distributed?',
    hole=0.5,
    color_discrete_sequence=px.colors.qualitative.Bold,
    template='plotly_dark',
    hover_data={'Description': True, 'Num_Recipients': True},
)
fig.update_traces(textinfo='label+percent+value',
                  texttemplate='%{label}<br>$%{value:,.0f}<br>(%{percent})')
fig.update_layout(height=520, title_font_size=20)
fig.show()

**📌 Insight:** **Game Championships (45%)** is the largest slice — $45M distributed across the 25 individual tournament prize pools. The **Club Championship** at 27% ($27M) is the second biggest incentive, explaining why major organizations compete across as many titles as possible. **Qualifiers** ($15M, 15%) ensures the prize pool reaches 200+ regional teams, not just the elite 272 who made it to Riyadh.


In [23]:
# ── Prize per recipient analysis ──
prize_dist['Prize_Per_Recipient'] = prize_dist['Amount_USD'] / prize_dist['Num_Recipients']

fig = px.bar(
    prize_dist.sort_values('Prize_Per_Recipient', ascending=False),
    x='Category',
    y='Prize_Per_Recipient',
    color='Category',
    title='💎 Average Prize per Recipient by Category',
    labels={'Prize_Per_Recipient': 'Avg Prize per Recipient (USD)', 'Category': ''},
    text='Prize_Per_Recipient',
    color_discrete_sequence=px.colors.qualitative.Prism,
    template='plotly_dark',
)
fig.update_traces(texttemplate='$%{y:,.0f}', textposition='outside')
fig.update_layout(height=420, title_font_size=18, showlegend=False)
fig.show()

**📌 Insight:** **Club Championship** and **Game Championships** have the highest per-recipient payouts — these are elite-level rewards for the best performers. **Qualifiers** serve 200+ teams with ~$75K per recipient — enough to sustain regional scenes but a fraction of what the Riyadh spotlight offers. This structure creates a clear incentive pyramid: qualify → compete → dominate.


In [24]:
# ── Tournament prize pools vs participant count ──
fig = px.scatter(
    tournaments,
    x='Num_Participants',
    y='Prize_Pool_USD',
    size='Duration_Days',
    color='Game_Type',
    hover_name='Game',
    title='📈 Prize Pool vs Participants (bubble = duration)',
    labels={'Num_Participants': 'Number of Participants', 'Prize_Pool_USD': 'Prize Pool (USD)'},
    color_discrete_sequence=px.colors.qualitative.Vivid,
    template='plotly_dark',
    size_max=35,
    log_y=False,
)
fig.update_layout(height=520, title_font_size=18, yaxis_tickformat='$,.0f')
fig.show()

**📌 Insight:** Prize pools don't scale linearly with participant count. Some of the **smallest fields** (Chess: 16 players, Dota 2: 16 teams) carry massive prize pools. The **outlier in participants** is Street Fighter 6 (48 players) and PUBG Mobile (24 teams) — wide-open fields that attract more qualifiers. **Racing (Rennsport)** has only 9 participants and a $500K pool — the most exclusive field per dollar.


---
## 9. 🤯 What Surprised Us in This Dataset? <a id='9'></a>

Every dataset has hidden stories that only emerge when you actually dig in. Here are the moments that made us stop and think.


In [25]:
print("=" * 60)
print("  SURPRISE #1: CHESS AT AN ESPORTS WORLD CUP")
print("=" * 60)
chess_row = tournaments[tournaments['Game'] == 'Chess'].iloc[0]
print(f"  Prize pool: ${chess_row['Prize_Pool_USD']:,}")
print(f"  Winner: {chess_row['Winner']}")
print(f"  Genre: {chess_row['Game_Type']}")
print(f"  Players: {chess_row['Num_Participants']}")
print()
print("  Magnus Carlsen — the world chess champion — competed")
print("  and won $1.5M at an esports tournament. That's a story")
print("  that blurs the line between traditional and digital sport.")

print()
print("=" * 60)
print("  SURPRISE #2: MIDDLE EAST CLUBS AT THE TOP")
print("=" * 60)
me_clubs = club_standings[club_standings['Region'] == 'Middle East']
print(me_clubs[['Organization', 'Rank', 'Total_Points', 'Tournament_Wins']].to_string(index=False))
print()
print("  Team Falcons (Saudi Arabia) ranked #1 overall —")
print("  ahead of legendary EU/NA orgs like Liquid and Vitality.")

print()
print("=" * 60)
print("  SURPRISE #3: MYANMAR IN THE MEDAL TABLE")
print("=" * 60)
myanmar = countries[countries['Country'] == 'Myanmar']
print(myanmar[['Country', 'Gold_Medals', 'Total_Medals', 'Total_Players', 'Top_Game']].to_string(index=False))
print()
print("  With just 16 players, Myanmar claimed Gold in PUBG Mobile —")
print("  Yangon Galacticos. Mobile gaming is levelling the global field.")

print()
print("=" * 60)
print("  SURPRISE #4: PRIZE MONEY AT $0 FOR MANY GOLD MEDALISTS")
print("=" * 60)
gold_zero = players[(players['Tournament_Place'] == 1) & (players['Prize_Earned_USD'] == 0)]
print(f"  Players with 1st place & $0 individual prize: {len(gold_zero)}")
print("  This confirms prizes flow through organizations, not")
print("  directly to players — a key structural feature of esports.")

  SURPRISE #1: CHESS AT AN ESPORTS WORLD CUP
  Prize pool: $1,500,000
  Winner: Magnus Carlsen (Team Liquid)
  Genre: Strategy
  Players: 16

  Magnus Carlsen — the world chess champion — competed
  and won $1.5M at an esports tournament. That's a story
  that blurs the line between traditional and digital sport.

  SURPRISE #2: MIDDLE EAST CLUBS AT THE TOP
  Organization  Rank  Total_Points  Tournament_Wins
  Team Falcons     1          3850                2
 Twisted Minds     4          1800                2
    Al Qadsiah    20           500                0
Geekay Esports    22           400                0

  Team Falcons (Saudi Arabia) ranked #1 overall —
  ahead of legendary EU/NA orgs like Liquid and Vitality.

  SURPRISE #3: MYANMAR IN THE MEDAL TABLE
Country  Gold_Medals  Total_Medals  Total_Players    Top_Game
Myanmar            1             2             16 PUBG Mobile

  With just 16 players, Myanmar claimed Gold in PUBG Mobile —
  Yangon Galacticos. Mobile gaming is lev

In [26]:
# ── Surprise visual: mobile vs PC prize pools ──
platform_prize = tournaments.groupby('Platform')['Prize_Pool_USD'].sum().reset_index()
platform_prize.columns = ['Platform', 'Total_Prize']
platform_prize = platform_prize.sort_values('Total_Prize', ascending=False)

fig = px.bar(
    platform_prize,
    x='Platform',
    y='Total_Prize',
    color='Platform',
    title='📱 vs 🖥️ Mobile vs PC Prize Pools — EWC 2025',
    labels={'Total_Prize': 'Total Prize Pool (USD)', 'Platform': ''},
    text='Total_Prize',
    color_discrete_sequence=['#00C9A7', '#FF4655', '#1E90FF', '#FFD700'],
    template='plotly_dark',
)
fig.update_traces(texttemplate='$%{y:,.0f}', textposition='outside')
fig.update_layout(height=400, title_font_size=18, showlegend=False)
fig.show()

**📌 Surprise:** **PC dominates prize money** — but **Mobile is a massive force** in participation and fan engagement. Games like PUBG Mobile, Free Fire, Mobile Legends, and Honor of Kings are uniquely dominant in Southeast Asia, South Asia, and the Middle East. EWC is one of the few events where a PC Dota 2 world champion and a PUBG Mobile team from Myanmar share the same stage.


---
## 10. 🔑 Key Findings & Takeaways <a id='10'></a>

Let's consolidate everything we've learned from EWC 2025.


In [27]:
summary = {
    '🏆 Total Prize Pool': f"${tournaments['Prize_Pool_USD'].sum():,.0f} (across 27 tournaments)",
    '🌍 Competing Nations': f"{countries.shape[0]} countries",
    '👤 Registered Players': f"{players.shape[0]} players in the roster dataset",
    '🥇 Top Country (medals)': f"South Korea — 12 medals (4G, 3S, 5B)",
    '🥇 Top Country (efficiency)': "Finland & Japan — highest medals-per-player",
    '🏢 Club Champion': "Team Falcons (Middle East) — 3,850 pts, $7M prize",
    '💰 Largest Prize Pool (single game)': f"Dota 2 / PUBG Mobile / Mobile Legends / HoK — $3M each",
    '⏱️ Longest Final': "Chess Grand Final — 240 minutes",
    '⚡ Shortest Final': "Tekken 8 / EA Sports FC — ~25–30 minutes",
    '🌟 Top Region (players)': "Asia & Europe together = ~65%+ of player pool",
    '🤯 Biggest Surprise': "Magnus Carlsen winning at an esports event",
}

print("\n" + "="*65)
print("  📋  EWC 2025 — DATASET SUMMARY & KEY FINDINGS")
print("="*65)
for k, v in summary.items():
    print(f"  {k}")
    print(f"       {v}")
    print()


  📋  EWC 2025 — DATASET SUMMARY & KEY FINDINGS
  🏆 Total Prize Pool
       $38,500,000 (across 27 tournaments)

  🌍 Competing Nations
       36 countries

  👤 Registered Players
       272 players in the roster dataset

  🥇 Top Country (medals)
       South Korea — 12 medals (4G, 3S, 5B)

  🥇 Top Country (efficiency)
       Finland & Japan — highest medals-per-player

  🏢 Club Champion
       Team Falcons (Middle East) — 3,850 pts, $7M prize

  💰 Largest Prize Pool (single game)
       Dota 2 / PUBG Mobile / Mobile Legends / HoK — $3M each

  ⏱️ Longest Final
       Chess Grand Final — 240 minutes

  ⚡ Shortest Final
       Tekken 8 / EA Sports FC — ~25–30 minutes

  🌟 Top Region (players)
       Asia & Europe together = ~65%+ of player pool

  🤯 Biggest Surprise
       Magnus Carlsen winning at an esports event



In [28]:
# ── Final visualization: Club Championship top 5 radar ──
top5 = club_standings.head(5)

categories = ['Total_Points', 'Prize_Money_USD', 'Tournament_Wins', 'Top_8_Finishes']
labels = ['Points', 'Prize ($M)', 'Wins', 'Top-8 Finishes']

fig = go.Figure()

for _, row in top5.iterrows():
    vals = [
        row['Total_Points'] / 100,          # scale to ~0-40
        row['Prize_Money_USD'] / 1_000_000, # in millions
        row['Tournament_Wins'] * 10,         # scale
        row['Top_8_Finishes'] * 5,           # scale
    ]
    vals_closed = vals + [vals[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=labels + [labels[0]],
        fill='toself',
        name=row['Organization'],
        opacity=0.6,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 40])),
    title='🕸️ Top 5 Clubs — Performance Radar (normalized)',
    template='plotly_dark',
    height=520,
    title_font_size=18,
)
fig.show()

---

## 🎓 Lessons & Next Steps

**What this dataset teaches us:**

1. **Esports is global — but Asia leads.** South Korea, China, and emerging Southeast Asian nations dominate, particularly in mobile and MOBA titles. The talent geography of esports is shifting.

2. **The Club Championship rewards breadth.** Team Falcons' success came from consistent top-8 finishes across multiple games, not just a single dominant title. Organizations must build multi-game rosters to compete.

3. **Mobile gaming is the great equalizer.** Nations like Myanmar, Indonesia, and Philippines punch well above their weight in mobile titles — lower hardware barriers mean wider talent pools.

4. **Prize money flows through organizations, not players.** Individual player `Prize_Earned_USD` fields are largely zero — the dollars flow to clubs first. This has implications for player power and negotiation.

5. **EWC is genuinely multimodal.** From Chess grandmasters to sim racers to MOBA teams — the event resists simple classification. That's its greatest strength and complexity.

**Next steps for deeper analysis:**
- 📊 Build a predictive model for Club Championship rank using tournament composition data
- 🔗 Enrich with historical EWC 2024 data to identify rising/falling organizations
- 🌐 Scrape live social following and viewership numbers for correlation analysis
- 📱 Deep-dive into mobile vs. PC game genre trends over the past 3 years of EWC

---

*If this notebook helped you understand the EWC 2025 dataset better, consider upvoting ⬆️ — it helps others discover it!*

*Dataset source: CC BY 4.0 — Fan/researcher compiled from public EWC 2025 results.*
